# Day 16 · 数据摄取

> 🎯 **今日学习目标**
> 1. 掌握 CSV/Excel 读取与编码处理
> 2. 掌握 JSON/SQL 数据读取
> 3. 掌握 API 数据拉取与数据导出

---

## 引入

📖 **数据不只是 CSV** — JSON、Excel、数据库、API 都是常见来源。
今天学习：**如何把外部数据变成 DataFrame**。

数据常见来源：
1. CSV 文件（最常见）
2. Excel 报表
3. JSON 接口 / 文件
4. SQL 数据库
5. HTTP API

---

## 第一讲 · CSV 与 Excel 读取

### 1.1 read_csv 参数详解

📖 **核心概念**：`read_csv` 是 Pandas 最常用的读取函数。

In [ ]:
# 导入 pandas 库
import pandas as pd
from io import StringIO

# 模拟 CSV 字符串（实际用 pd.read_csv('data.csv')）
csv_data = """姓名,年龄,城市
张三,25,北京
李四,30,上海
王五,28,深圳
赵六,35,广州"""

# read_csv 常用参数：
#   sep=','           分隔符，默认逗号
#   encoding='utf-8'  编码（中文常用 gbk）
#   index_col=0       把第 0 列当作索引
#   usecols=['姓名']  只读指定列
#   nrows=10          只读前 10 行
#   skiprows=2        跳过前 2 行
#   na_values=['NA']  把哪些值当作缺失值

# 从字符串读取 CSV
df = pd.read_csv(StringIO(csv_data))

# 查看原始数据
print("=== 原始数据 ===")
print(df.head())

💡 **进阶用法演示**：

In [ ]:
# 演示：跳过前 2 行，只选两列
df2 = pd.read_csv(
    StringIO(csv_data),
    skiprows=2,               # 跳过前 2 行
    usecols=['姓名', '年龄']  # 只读指定列
)

# 打印结果
print("=== 跳过前 2 行 ===")
print(df2)
print(df2.info())

🔴 **常见错误：编码问题**

中文 Windows CSV 常用 GBK，先试 utf-8 再试 gbk。

In [ ]:
# 编码处理示例（伪代码）
# try:
#     # 先尝试 utf-8 编码
#     df = pd.read_csv("data.csv", encoding="utf-8")
# except UnicodeDecodeError:
#     # utf-8 失败时尝试 gbk
#     df = pd.read_csv("data.csv", encoding="gbk")

# 模拟一个 DataFrame 演示
df = pd.DataFrame({
    "城市": ["北京", "上海"],
    "人口": [2154, 2424]
})
print(df)

### 1.2 read_excel 读取 Excel

📖 **读取 Excel 文件**，支持多 Sheet。

In [ ]:
# read_excel 常用参数（注释展示）：
#   sheet_name=0        第 1 个 sheet（int）
#   sheet_name='Sheet2' 按名字取
#   sheet_name=None     全部 sheet，返回字典
#   usecols / header    用法同 read_csv

# 模拟：构造一个含两个 sheet 的 Excel 文件
import os
with pd.ExcelWriter('demo.xlsx') as writer:
    # 写入数学成绩 sheet
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [88, 92]}
    ).to_excel(writer, sheet_name='数学', index=False)
    # 写入语文成绩 sheet
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [76, 85]}
    ).to_excel(writer, sheet_name='语文', index=False)

# 读取全部 sheet
all_sheets = pd.read_excel('demo.xlsx', sheet_name=None)

# 打印结果
print("=== 模拟多 sheet 读取 ===")
print(all_sheets)

# 清理临时文件
os.remove('demo.xlsx')

✏️ **练习 1-1**：用 `read_csv` 读取，跳过前 2 行说明行。

In [ ]:
import pandas as pd
from io import StringIO

# 含中文的 CSV 字符串（前 2 行是说明）
csv_data = """这是说明行
第二行也是标题
姓名,年龄,城市
张三,25,北京
李四,30,上海"""

# TODO: 用 read_csv 读取，跳过前 2 行
pass

In [ ]:
# 参考答案
import pandas as pd
from io import StringIO

csv_data = """这是说明行
第二行也是标题
姓名,年龄,城市
张三,25,北京
李四,30,上海"""

# 将字符串转为文件对象
csv_file = StringIO(csv_data)
# 跳过前 2 行读取
df = pd.read_csv(csv_file, skiprows=2)
print(df)

✏️ **练习 1-2**：读取 Excel 里指定 sheet，只取部分列。

In [ ]:
import pandas as pd

# TODO: 读取 demo.xlsx 里 sheet '语文'
#       只取姓名和成绩列
# 提示：先用 ExcelWriter 创建文件再读
pass

In [ ]:
# 参考答案
import pandas as pd
import os

# 先构造临时 Excel
with pd.ExcelWriter('demo.xlsx') as writer:
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [88, 92]}
    ).to_excel(writer, sheet_name='数学', index=False)
    pd.DataFrame(
        {'姓名': ['张三', '李四'], '成绩': [76, 85]}
    ).to_excel(writer, sheet_name='语文', index=False)

# 读取指定 sheet 和列
df = pd.read_excel(
    'demo.xlsx',
    sheet_name='语文',
    usecols=['姓名', '成绩']
)
print(df)

# 清理文件
os.remove('demo.xlsx')

✅ **检查**：你能处理中文编码和跳行读取了吗？

---

## 第二讲 · JSON 与数据库读取

### 2.1 read_json 从 JSON 读取

📖 **JSON 是 API 最常用的数据格式**。

In [ ]:
# 导入 pandas
import pandas as pd

# 定义 JSON 字符串
json_str = '''[
    {"姓名": "张三", "年龄": 25, "城市": "北京"},
    {"姓名": "李四", "年龄": 30, "城市": "上海"},
    {"姓名": "王五", "年龄": 28, "城市": "深圳"}
]'''

# read_json 常用参数：
#   orient='records'  JSON 数组格式
#   force_ascii=False 中文不转义
#   lines=True         每行一条记录

# 从 JSON 字符串读取为 DataFrame
df = pd.read_json(json_str)

# 打印结果
print("=== read_json：从 JSON 字符串读入 ===")
print(df)

💡 **嵌套 JSON 处理**：

In [ ]:
# 嵌套 JSON：每个学生有 scores 字段（dict）
json_nested = '''[
    {"姓名": "张三", "scores": {"数学": 88, "语文": 76}},
    {"姓名": "李四", "scores": {"数学": 92, "语文": 85}}
]'''

# 读取嵌套 JSON
df_nest = pd.read_json(json_nested)

# 提取数学成绩到新列
df_nest['数学'] = df_nest['scores'].apply(
    lambda x: x['数学']
)

# 只保留需要的列
df_nest = df_nest[['姓名', '数学']]
print(df_nest)

### 2.2 read_sql 从数据库读取

📖 **从 SQLite 数据库读取数据**。

In [ ]:
# 导入所需库
import sqlite3
import pandas as pd

# 用 with 自动关闭连接
with sqlite3.connect(':memory:') as conn:
    # 创建游标
    cursor = conn.cursor()
    # 创建学生表
    cursor.execute(
        'CREATE TABLE students '
        '(id INTEGER PRIMARY KEY, '
        '姓名 TEXT, 年龄 INTEGER)'
    )
    # 批量插入数据
    cursor.executemany(
        'INSERT INTO students (姓名, 年龄) '
        'VALUES (?, ?)',
        [('张三', 25), ('李四', 30), ('王五', 28)]
    )
    # 提交事务
    conn.commit()

    # 用 read_sql 读整张表
    df_all = pd.read_sql(
        'SELECT * FROM students', conn
    )
    print("=== 读取整张表 ===")
    print(df_all)

    # 带条件查询
    df_filtered = pd.read_sql(
        'SELECT * FROM students WHERE 年龄 >= 30',
        conn
    )
    print("\n=== 条件查询 ===")
    print(df_filtered)
# with 结束，连接自动关闭

🔴 **常见错误：忘记关闭连接**

用完必须 `conn.close()` 或用 `with` 语句。

In [ ]:
# 错误示范（不要这样写）：
# conn = sqlite3.connect('mydb.db')
# df = pd.read_sql('SELECT * FROM t', conn)
# 忘记 close → 可能导致文件锁死！

# 正确写法：
# conn = sqlite3.connect('mydb.db')
# df = pd.read_sql('SELECT * FROM t', conn)
# conn.close()  # 必须关闭

# 最佳写法（推荐）：
# with sqlite3.connect('mydb.db') as conn:
#     df = pd.read_sql('SELECT * FROM t', conn)
# 自动关闭，无需手动 close
print("记住：连接用完必须关闭！")

✏️ **练习 2-3**：从嵌套 JSON 读取并提取数学成绩。

In [ ]:
import pandas as pd

# 嵌套 JSON：每个学生有 scores 字段
json_str = '''[
    {"姓名": "张三", "scores": {"数学": 88, "语文": 76}},
    {"姓名": "李四", "scores": {"数学": 92, "语文": 85}}
]'''

# TODO: 读入并提取数学成绩到新列
pass

In [ ]:
# 参考答案
import pandas as pd

json_str = '''[
    {"姓名": "张三", "scores": {"数学": 88, "语文": 76}},
    {"姓名": "李四", "scores": {"数学": 92, "语文": 85}}
]'''

# 读取 JSON
df = pd.read_json(json_str)
# 提取数学成绩
df['数学'] = df['scores'].apply(
    lambda x: x['数学']
)
# 只保留姓名和数学列
df = df[['姓名', '数学']]
print(df)

✏️ **练习 2-4**：创建内存数据库，用 read_sql 查询。

In [ ]:
import sqlite3
import pandas as pd

# TODO: 用 with 创建内存数据库
#       创建 users 表，插入 2 条数据，查询
pass

In [ ]:
# 参考答案
import sqlite3
import pandas as pd

# 使用 with 管理连接
with sqlite3.connect(':memory:') as conn:
    # 获取游标
    cursor = conn.cursor()
    # 创建表
    cursor.execute(
        'CREATE TABLE users '
        '(id INTEGER PRIMARY KEY, '
        '姓名 TEXT, 年龄 INTEGER)'
    )
    # 插入数据
    cursor.executemany(
        'INSERT INTO users (姓名, 年龄) '
        'VALUES (?, ?)',
        [('张三', 25), ('李四', 30)]
    )
    # 提交
    conn.commit()
    # 查询
    df = pd.read_sql('SELECT * FROM users', conn)
    print(df)

✅ **检查**：你掌握了 JSON 和 SQL 的读取方法吗？

---

## 第三讲 · API 数据拉取与数据导出

### 3.1 从 API 拉取数据

📖 **API 是获取网络数据的核心方式**。

In [ ]:
# 导入 requests 库（需要安装）
# pip install requests
import requests
import pandas as pd

# 请求公开测试 API
url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url)

# 检查状态码
if response.status_code == 200:
    # 解析 JSON 数据
    data = response.json()
    # 转为 DataFrame
    df = pd.DataFrame(data)
    # 只看部分列
    print("=== 请求用户接口 ===")
    print(df[['id', 'name', 'email']].head(3))
    print(f"返回了 {len(df)} 条数据")
else:
    print(f"请求失败: {response.status_code}")

💡 **带参数的 API 请求**：

In [ ]:
# 带 params 的请求示例
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"_limit": 3}  # 限制返回 3 条
)

# 判断是否成功
if response.status_code == 200:
    # 解析并转为 DataFrame
    df = pd.DataFrame(response.json())
    print(df[['id', 'title']])

# 注意：此代码需要网络连接

### 3.2 数据导出

📖 **将处理好的数据保存到不同格式**。

In [ ]:
# 导入库
import pandas as pd
import sqlite3
import os

# 准备示例数据
df = pd.DataFrame({
    '姓名': ['张三', '李四', '王五'],
    '年龄': [25, 30, 28],
    '城市': ['北京', '上海', '深圳']
})

# 导出为 CSV（不保存索引）
df.to_csv(
    'out.csv', index=False, encoding='utf-8'
)
print("=== 导出 CSV 成功 ===")

# 导出为 JSON（记录格式，保留中文）
df.to_json(
    'out.json',
    orient='records',
    force_ascii=False
)
print("=== 导出 JSON 成功 ===")

# 导出到 SQLite 数据库
with sqlite3.connect('out.db') as conn:
    df.to_sql(
        'people', conn,
        if_exists='replace',  # 表存在则替换
        index=False
    )
    count = pd.read_sql(
        'SELECT COUNT(*) FROM people', conn
    )
    print("=== 导出 SQL 成功 ===")

# 清理临时文件
os.remove('out.csv')
os.remove('out.json')
os.remove('out.db')

🔴 **注意**：`index=False` 避免导出多余索引列。

✏️ **练习 3-5**：从 API 拉取帖子数据并导出 CSV。

> 提示：`/posts` 接口，需要网络连接

In [ ]:
import requests
import pandas as pd

# TODO: 调用 /posts 接口，取前 5 条
#       只保留 userId 和 title，导出 CSV
# 注意：需要网络连接
pass

In [ ]:
# 参考答案
import requests
import pandas as pd

# 请求 API（限制 5 条）
url = 'https://jsonplaceholder.typicode.com/posts'
resp = requests.get(url, params={'_limit': 5})

# 判断是否成功
if resp.status_code == 200:
    # 解析 JSON 并转 DataFrame
    df = pd.DataFrame(resp.json())
    # 只取两列
    df = df[['userId', 'title']]
    # 导出 CSV
    df.to_csv('posts.csv', index=False)
    print(df)
else:
    print('请求失败')

✏️ **练习 3-6**：读取 CSV 再导出为 JSON 格式。

In [ ]:
import pandas as pd

# TODO: 读 posts.csv，导出为 JSON
#       使用 orient='records'
#       force_ascii=False
pass

In [ ]:
# 参考答案
import pandas as pd

# 读取 CSV
df = pd.read_csv('posts.csv')
# 导出为 JSON
df.to_json(
    'posts.json',
    orient='records',
    force_ascii=False,
    indent=2
)
print("导出 JSON 完成!")
print(df)

✅ **检查**：你能从 API 获取数据并导出了吗？

---

## ⭐ 挑战题 · 综合练习

⭐ **挑战：从多种来源读取数据并合并分析**

场景：两份数据
- JSON 字符串（学生信息）
- SQLite 数据库（成绩信息）

任务：合并后计算每个学生的平均分。

In [ ]:
import requests
import pandas as pd

# 挑战题：从 API 拉取 /users
#       清洗缺失的 'email' 字段
#       用 'unknown' 填充
#       导出为 'users_clean.csv'
url = 'https://jsonplaceholder.typicode.com/users'

# TODO: 完成拉取、清洗、导出
# 注意：需要网络连接
pass

In [ ]:
# 参考答案
import requests
import pandas as pd
import os

url = 'https://jsonplaceholder.typicode.com/users'

# 1. 拉取数据
response = requests.get(url)
df = pd.DataFrame(response.json())

# 2. 清洗：email 缺失用 'unknown' 填充
if 'email' in df.columns:
    df['email'] = df['email'].fillna('unknown')

# 3. 导出 CSV
df.to_csv(
    'users_clean.csv', index=False, encoding='utf-8'
)

print("=== 拉取、清洗、导出完成 ===")
print(df[['id', 'name', 'email']].head(3))

# 清理
os.remove('users_clean.csv')

⭐ **进阶挑战：JSON + SQL 合并分析**

In [ ]:
# TODO: 从 JSON 和 SQL 读取数据并合并
import pandas as pd
import sqlite3

# JSON 数据（学生信息）
json_str = '''[
    {"id": 1, "name": "张三"},
    {"id": 2, "name": "李四"}
]'''

# SQL 数据库（成绩信息）
conn = sqlite3.connect(':memory:')
conn.execute(
    'CREATE TABLE scores (id INTEGER, score INTEGER)'
)
conn.execute('INSERT INTO scores VALUES (1, 85)')
conn.execute('INSERT INTO scores VALUES (1, 90)')
conn.execute('INSERT INTO scores VALUES (2, 78)')
conn.execute('INSERT INTO scores VALUES (2, 88)')
conn.commit()

# TODO: 读取两者，合并，计算平均分
pass
conn.close()

In [ ]:
# 参考答案
import pandas as pd
import sqlite3

# JSON 数据
json_str = '''[
    {"id": 1, "name": "张三"},
    {"id": 2, "name": "李四"}
]'''
# 读取 JSON
df_stu = pd.read_json(json_str)

# SQL 数据库
conn = sqlite3.connect(':memory:')
conn.execute(
    'CREATE TABLE scores (id INTEGER, score INTEGER)'
)
conn.execute('INSERT INTO scores VALUES (1, 85)')
conn.execute('INSERT INTO scores VALUES (1, 90)')
conn.execute('INSERT INTO scores VALUES (2, 78)')
conn.execute('INSERT INTO scores VALUES (2, 88)')
conn.commit()

# 读取 SQL
df_sco = pd.read_sql('SELECT * FROM scores', conn)

# 按 id 分组计算平均分
df_avg = df_sco.groupby('id')['score'].mean()
df_avg = df_avg.reset_index()

# 合并学生姓名
result = pd.merge(df_stu, df_avg, on='id')
print(result)

# 关闭连接
conn.close()

---

## 📝 今日总结

| 数据源 | 方法 | 关键点 |
|--------|------|--------|
| CSV | `read_csv()` | 注意编码 |
| Excel | `read_excel()` | 多 Sheet |
| JSON | `read_json()` | 字符串或文件 |
| SQL | `read_sql()` | 关闭连接 |
| API | `requests.get()` | 需网络 |
| 导出 | `to_csv/to_json/to_sql` | `index=False`|